# P9-1 · Đọc và tái hiện paper

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

In [ ]:
!pip install -q peft

## Lý thuyết

## Bài tập 5: Kiểm tra một con số {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
from transformers import AutoModelForSequenceClassification, DistilBertConfig
from peft import LoraConfig, get_peft_model

# from_config: đúng kiến trúc distilbert-base-uncased nhưng trọng số ngẫu nhiên, không cần tải;
# đủ để đếm tham số. Train thật thì dùng from_pretrained("distilbert-base-uncased", num_labels=2)
cfg = DistilBertConfig(num_labels=2)
full = AutoModelForSequenceClassification.from_config(cfg)
n_full = sum(p.numel() for p in full.parameters() if p.requires_grad)

lora = get_peft_model(AutoModelForSequenceClassification.from_config(cfg),
                      LoraConfig(r=8, lora_alpha=16, target_modules=["q_lin", "v_lin"], task_type="SEQ_CLS"))
n_lora = sum(p.numel() for p in lora.parameters() if p.requires_grad)
print(f"full {n_full:,} | LoRA {n_lora:,} | tỉ lệ {n_lora / n_full:.2%}")

full 66,955,010 | LoRA 739,586 | tỉ lệ 1.10%


## Bài tập 6: Cải thiện này có vượt qua nhiễu không {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np
from scipy.stats import ttest_ind
baseline = np.array([77.6, 78.3, 77.2, 78.0, 77.9])
moi_A = np.array([78.1, 77.5, 78.9, 77.8, 78.2])          # phương pháp trong bài báo
moi_B = np.array([79.4, 79.9, 79.1, 80.2, 79.6])          # một phương pháp khác
# TODO: in mean ± std; p-value Welch cho A và cho B so với baseline

In [ ]:
for ten, x in [("baseline", baseline), ("A", moi_A), ("B", moi_B)]:
    print(f"{ten:8s}: {x.mean():.2f} ± {x.std(ddof=1):.2f}")
for ten, x in [("A", moi_A), ("B", moi_B)]:
    t = ttest_ind(x, baseline, equal_var=False)
    print(f"{ten} so với baseline: +{x.mean() - baseline.mean():.2f}, p = {t.pvalue:.3f}")

baseline: 77.80 ± 0.42
A       : 78.10 ± 0.52
B       : 79.64 ± 0.43
A so với baseline: +0.30, p = 0.348
B so với baseline: +1.84, p = 0.000


## Bài tập 7: Đọc bảng ablation {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import itertools
ket_qua = {  # (H, R, Q): recall@5
    (0, 0, 0): 0.62, (1, 0, 0): 0.71, (0, 1, 0): 0.70, (0, 0, 1): 0.66,
    (1, 1, 0): 0.78, (1, 0, 1): 0.73, (0, 1, 1): 0.72, (1, 1, 1): 0.79}
ten = ["H", "R", "Q"]
# TODO: dong_gop[ten] = trung bình (kq với thành phần = 1) - (kq với thành phần = 0) qua 4 cặp

In [ ]:
dong_gop = {}
for i, t in enumerate(ten):
    hieu = []
    for k, v in ket_qua.items():
        if k[i] == 1:
            k0 = tuple(0 if j == i else x for j, x in enumerate(k))
            hieu.append(v - ket_qua[k0])
    dong_gop[t] = sum(hieu) / len(hieu)
print({k: round(v, 3) for k, v in dong_gop.items()})
print("Q khi đã có H và R:", round(ket_qua[(1, 1, 1)] - ket_qua[(1, 1, 0)], 3))

{'H': 0.078, 'R': 0.068, 'Q': 0.023}
Q khi đã có H và R: 0.01
